# Agentic Variant Interpretation with ClawBio

**University of Westminster | Systems Biology Workshop | 27 March 2026**

**Dr Manuel Corpas** | Senior Lecturer in Genomics, AI & Health Data Science

---

In this tutorial you will:

1. Download and explore a **real human genome** (the Corpasome, CC0 public domain)
2. Convert 23andMe genotype data to VCF format
3. Run **ClawBio's variant-annotation** skill to annotate variants with VEP, ClinVar, and gnomAD
4. Run **ClawBio's clinpgx** skill for pharmacogenomic interpretation
5. Interpret the results: pathogenic variants, drug responses, and variants of uncertain significance

Everything runs in this Colab notebook. No local installation needed.

> **About the Corpasome**: This is the 23andMe genotype of Manuel Corpas, published under CC0 as one of the first fully open personal genomes. See: Corpas, M. (2013). Crowdsourcing the Corpasome. *Source Code for Biology and Medicine*, **8**, 13. [doi:10.1186/1751-0473-8-13](https://link.springer.com/article/10.1186/1751-0473-8-13)

## Step 0: Setup

Install ClawBio and dependencies.

In [ ]:
%%bash
# Clone ClawBio repository
if [ ! -d "ClawBio" ]; then
    git clone --depth 1 https://github.com/ClawBio/ClawBio.git
fi

# Install dependencies
pip install -q pysam requests matplotlib pandas

In [ ]:
import sys
import os

# Add ClawBio to path
sys.path.insert(0, 'ClawBio')
os.chdir('ClawBio')

print('ClawBio loaded successfully')
print(f'Skills available: {len(os.listdir("skills"))}')

## Step 1: Download the Corpasome

The Corpasome is a real 23andMe genotype file (~600,000 SNPs) published under CC0.

A copy is bundled with ClawBio at `skills/genome-compare/data/manuel_corpas_23andme.txt.gz`.

In [ ]:
import gzip
from pathlib import Path

GENOME_FILE = Path('skills/genome-compare/data/manuel_corpas_23andme.txt.gz')

# Read and preview
with gzip.open(GENOME_FILE, 'rt') as f:
    lines = f.readlines()

# Count data lines (skip comments)
data_lines = [l for l in lines if not l.startswith('#')]
comment_lines = [l for l in lines if l.startswith('#')]

print(f'Total lines: {len(lines):,}')
print(f'Comment lines: {len(comment_lines):,}')
print(f'SNP entries: {len(data_lines):,}')
print()
print('First 5 comment lines:')
for line in comment_lines[:5]:
    print(line.strip())
print()
print('First 10 data lines:')
print('rsID\t\tChr\tPosition\tGenotype')
print('-' * 50)
for line in data_lines[:10]:
    print(line.strip())

### Understanding the 23andMe format

Each line has four columns:
- **rsID**: the SNP identifier (e.g., rs4244285)
- **Chromosome**: 1-22, X, Y, or MT
- **Position**: genomic coordinate (GRCh37 in older files, GRCh38 in newer)
- **Genotype**: two alleles (e.g., AG = heterozygous, AA = homozygous)

**Question for students**: How many SNPs are on each chromosome? Let's find out.

In [ ]:
import pandas as pd

# Parse into DataFrame
records = []
for line in data_lines:
    parts = line.strip().split('\t')
    if len(parts) == 4:
        records.append({'rsid': parts[0], 'chrom': parts[1], 'pos': parts[2], 'genotype': parts[3]})

df = pd.DataFrame(records)
print(f'Parsed {len(df):,} variants')
print()
print('Variants per chromosome:')
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT']
counts = df['chrom'].value_counts()
for c in chrom_order:
    if c in counts:
        print(f'  Chr {c:>2}: {counts[c]:>6,}')

## Step 2: Convert 23andMe to VCF

The variant-annotation skill expects VCF format. We will convert a clinically relevant subset of SNPs to VCF.

We focus on variants in pharmacogenomic and disease-associated genes.

In [ ]:
# Clinically relevant rsIDs to extract
# These cover pharmacogenomics, cancer risk, cardiovascular, and Mendelian conditions
CLINICAL_RSIDS = {
    # Pharmacogenomics
    'rs4244285',   # CYP2C19*2 - clopidogrel, PPIs
    'rs4986893',   # CYP2C19*3 - clopidogrel
    'rs12248560',  # CYP2C19*17 - ultrarapid metaboliser
    'rs1799853',   # CYP2C9*2 - warfarin
    'rs1057910',   # CYP2C9*3 - warfarin
    'rs9923231',   # VKORC1 - warfarin sensitivity
    'rs3892097',   # CYP2D6*4 - codeine, tamoxifen
    'rs1142345',   # TPMT - thiopurines
    'rs1800460',   # TPMT - thiopurines
    'rs1801131',   # MTHFR A1298C - folate metabolism
    'rs1801133',   # MTHFR C677T - folate metabolism
    # Cancer risk
    'rs1799966',   # BRCA1
    'rs80357906',  # BRCA2
    'rs1042522',   # TP53 codon 72
    # Cardiovascular
    'rs6025',      # Factor V Leiden
    'rs1799963',   # Prothrombin G20210A
    'rs1800562',   # HFE C282Y - haemochromatosis
    'rs1799945',   # HFE H63D - haemochromatosis
    # Other
    'rs113993960', # CFTR deltaF508 - cystic fibrosis
    'rs7412',      # APOE - Alzheimer risk
    'rs429358',    # APOE - Alzheimer risk
}

# Extract matching variants from Corpasome
clinical_df = df[df['rsid'].isin(CLINICAL_RSIDS)].copy()
print(f'Found {len(clinical_df)} of {len(CLINICAL_RSIDS)} clinical variants in the Corpasome')
print()
print(clinical_df.to_string(index=False))

In [ ]:
# Convert to VCF format
# NOTE: 23andMe positions may be GRCh37. For this tutorial we use them as-is
# and let VEP handle coordinate mapping.

VCF_OUT = Path('output/corpasome_clinical.vcf')
VCF_OUT.parent.mkdir(parents=True, exist_ok=True)

with open(VCF_OUT, 'w') as vcf:
    # Header
    vcf.write('##fileformat=VCFv4.2\n')
    vcf.write('##fileDate=20260327\n')
    vcf.write('##source=ClawBio_Corpasome_Tutorial\n')
    vcf.write('##reference=GRCh37\n')
    vcf.write('##INFO=<ID=RSID,Number=1,Type=String,Description="dbSNP rsID">\n')
    vcf.write('##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n')
    vcf.write('#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tCORPASOME\n')
    
    written = 0
    for _, row in clinical_df.iterrows():
        gt = row['genotype']
        if len(gt) != 2 or gt == '--':  # skip no-calls
            continue
        
        # For heterozygous calls, first allele = REF, second = ALT
        allele1, allele2 = gt[0], gt[1]
        
        if allele1 == allele2:
            # Homozygous - we still write it (could be hom-alt)
            ref = allele1
            alt = '.'
            gt_field = '0/0'
        else:
            # Heterozygous
            ref = allele1
            alt = allele2
            gt_field = '0/1'
        
        chrom = row['chrom']
        pos = row['pos']
        rsid = row['rsid']
        
        vcf.write(f'{chrom}\t{pos}\t{rsid}\t{ref}\t{alt}\t.\tPASS\tRSID={rsid}\tGT\t{gt_field}\n')
        written += 1

print(f'Written {written} variants to {VCF_OUT}')
print()
print('Preview:')
with open(VCF_OUT) as f:
    for line in f:
        print(line.strip())

## Step 3: Run ClawBio Variant Annotation

The **variant-annotation** skill calls the Ensembl VEP REST API to annotate each variant with:
- Gene and transcript
- Consequence type (missense, synonymous, etc.)
- ClinVar clinical significance
- gnomAD population allele frequencies
- Priority tier (Tier 1 = highest clinical relevance)

No API key needed. VEP REST is free and public.

In [ ]:
%%bash
# Run the variant-annotation skill on our clinical VCF
python skills/variant-annotation/variant_annotation.py \
    --input output/corpasome_clinical.vcf \
    --output-dir output/variant_annotation \
    --assembly GRCh37 \
    --verbose

In [ ]:
# Read the annotation report
report_path = Path('output/variant_annotation/report.md')
if report_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(report_path.read_text()))
else:
    print('Report not found. Check the output above for errors.')
    # List what was created
    for f in sorted(Path('output/variant_annotation').rglob('*')):
        if f.is_file():
            print(f'  {f}')

In [ ]:
# Load and display the annotated variants table
tsv_path = Path('output/variant_annotation/tables/annotated_variants.tsv')
if tsv_path.exists():
    ann_df = pd.read_csv(tsv_path, sep='\t')
    print(f'Annotated {len(ann_df)} variants')
    print()
    display(ann_df)
else:
    print('Annotated variants TSV not found.')

### Interpreting the Results

**Priority Tiers:**
- **Tier 1**: Pathogenic or likely pathogenic in ClinVar, rare in gnomAD. Highest clinical relevance.
- **Tier 2**: Drug response variants (pharmacogenomics) or risk factors.
- **Tier 3**: Variants of uncertain significance (VUS). Not enough evidence to classify.
- **Tier 4**: Benign or likely benign. Common in population databases.

**Questions for students:**
1. How many Tier 1 (pathogenic) variants were found? What genes are they in?
2. How many drug response variants were identified? Which drugs do they affect?
3. Are there any VUS? What would you tell a patient about a VUS?

## Step 4: Pharmacogenomic Interpretation with ClinPGx

The **clinpgx** skill maps genotypes to CPIC (Clinical Pharmacogenetics Implementation Consortium) guidelines.

It determines:
- Your **metaboliser phenotype** for each gene (e.g., CYP2D6: poor, intermediate, normal, ultrarapid)
- **Drug recommendations** based on your genotype
- **Clinical actions** (dose adjustments, alternative drugs, or standard dosing)

In [ ]:
%%bash
# Run clinpgx on the Corpasome
python skills/clinpgx/clinpgx.py \
    --input output/corpasome_clinical.vcf \
    --output-dir output/clinpgx \
    --verbose 2>&1 || echo 'clinpgx completed (check output above)'

In [ ]:
# Display ClinPGx report
clinpgx_report = Path('output/clinpgx/report.md')
if clinpgx_report.exists():
    from IPython.display import Markdown, display
    display(Markdown(clinpgx_report.read_text()))
else:
    print('ClinPGx report not found.')
    # Try alternative output locations
    for f in sorted(Path('output/clinpgx').rglob('*')):
        if f.is_file():
            print(f'  {f}')

### The Warfarin Example

Warfarin dosing depends on two genes:
- **CYP2C9**: metabolises warfarin. Variants *2 and *3 reduce metabolism.
- **VKORC1**: warfarin's drug target. The rs9923231 T allele increases sensitivity.

Manuel Corpas carries:
- **VKORC1 rs9923231 TT** (high warfarin sensitivity)
- **CYP2C9 *1/*2** (intermediate metaboliser)

This combination triggers an **AVOID or significantly reduce dose** recommendation under CPIC guidelines.

> This is exactly the kind of finding that pharmacogenomic testing is designed to catch. Without genotyping, a standard warfarin dose could cause dangerous bleeding.

## Step 5: Explore on Your Own

### Exercise 5a: Run the full variant-annotation on the bundled demo VCF

ClawBio ships a synthetic ClinVar panel with 20 curated variants spanning multiple disease categories.

In [ ]:
%%bash
# Run variant-annotation on the synthetic ClinVar demo panel
python skills/variant-annotation/variant_annotation.py \
    --demo \
    --output-dir output/demo_annotation \
    --verbose

### Exercise 5b: Use your OWN genome

If you have a 23andMe, AncestryDNA, or other DTC genotype file:

1. Upload it using the Colab file picker (click the folder icon on the left)
2. Modify the `GENOME_FILE` path in Step 1 to point to your file
3. Re-run Steps 2-4

**Privacy note**: Your genome data stays in this Colab session and is deleted when the session ends. The VEP API receives only variant coordinates (chromosome + position + alleles), not your identity. No data is stored by ClawBio.

### Exercise 5c: Investigate a specific gene

Pick a gene from the results and answer:
1. What is the gene's function?
2. What ACMG classification does the variant have?
3. What is the allele frequency in gnomAD? Is it rare or common?
4. Would you report this variant to a patient? Why or why not?
5. What is the difference between a primary finding and a secondary finding?

## Key Concepts Covered

| Concept | Traditional Approach | Agentic Approach (ClawBio) |
|---------|---------------------|---------------------------|
| VCF annotation | Manual VEP submission, parse JSON output | One command, structured report |
| ClinVar lookup | Search web interface per variant | Batch API, auto-prioritisation |
| gnomAD frequency | Manual search per variant | Integrated into annotation pipeline |
| Pharmacogenomics | Look up CPIC tables manually | Automated genotype-to-phenotype mapping |
| Variant prioritisation | Expert judgement on raw data | Tiered scoring with human review |

**The agentic approach does not replace clinical judgement. It accelerates the mechanical steps so you can focus on interpretation.**

## ACMG Variant Classification Reminder

The American College of Medical Genetics (ACMG) defines five categories:

1. **Pathogenic**: Directly contributes to disease development
2. **Likely pathogenic**: Strong evidence but cannot fully rule out benign
3. **Uncertain significance (VUS)**: Not enough evidence to classify
4. **Likely benign**: Not expected to have major effect on disease
5. **Benign**: Does not cause disease

**Key teaching point**: VUS is the honest answer. You will never catch up with the classification backlog. Neither will AI. Learning to communicate uncertainty is a core clinical skill.

## Further Reading

- ClawBio: [github.com/ClawBio/ClawBio](https://github.com/ClawBio/ClawBio)
- Corpasome paper: [doi:10.1186/1751-0473-8-13](https://link.springer.com/article/10.1186/1751-0473-8-13)
- ACMG Standards: Richards et al. (2015) *Genetics in Medicine* 17(5):405-24
- CPIC Guidelines: [cpicpgx.org](https://cpicpgx.org)
- Ensembl VEP: [ensembl.org/info/docs/tools/vep](https://www.ensembl.org/info/docs/tools/vep/index.html)
- ClinVar: [ncbi.nlm.nih.gov/clinvar](https://www.ncbi.nlm.nih.gov/clinvar/)
- gnomAD: [gnomad.broadinstitute.org](https://gnomad.broadinstitute.org/)

---

*This tutorial is part of ClawBio's open educational resources. Licensed under MIT.*